# Ch12. Advanced Forecasting Methods
**Forecasting: Principles & Practice (Python Edition)**
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

The cells below follow the slide deck *Ch12. Advanced forecasting methods* (updated, 57 slides) and are meant to be run from top to bottom: later cells reuse objects created in earlier ones. Cells marked *figure code* reproduce a chart whose code is not shown on the slide.

Note: the dynamic harmonic regression for the Victorian electricity data ([Slide 21], `AutoARIMA` with 39 regressors on 17,518 half-hours) takes about 20 minutes; the bagging loop ([Slide 53], 100 `AutoETS` fits) a few minutes.

In [ ]:
%pip install -q statsforecast utilsforecast statsmodels prophet matplotlib

In [ ]:
BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"  # datasets in this repository (data/)

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os; os.environ["NIXTLA_ID_AS_COL"] = "true"
from functools import partial, reduce
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import MSTL, AutoARIMA, AutoETS
from statsforecast.models import AutoTBATS
from utilsforecast.feature_engineering import fourier
from utilsforecast.feature_engineering import pipeline
from utilsforecast.preprocessing import fill_gaps
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, mape, mase
from statsmodels.graphics.tsaplots import plot_acf
plt.rcParams.update({"figure.figsize": (7, 3.5)})
np.random.seed(1)   # blocked bootstrap (Section 12.4) uses random block starts

## Data
The slides for the weekly US gasoline (slide 4), Turkish electricity (slide 8) and Victorian electricity (slide 18) figures show no code. The series are loaded here; the other datasets are read in the slide cells where they appear.

In [ ]:
gas = pd.read_csv(BASE + "us_gasoline.csv", parse_dates=["ds"])          # weekly, 1991-2017
turkey = pd.read_csv(BASE + "turkey_elec.csv", header=None)[0] / 1e3    # daily GW, no header
turkey.index = pd.date_range("2000-01-01", periods=len(turkey), freq="D")
vic_elec = pd.read_csv(BASE + "vic_elec.csv", parse_dates=["ds"])       # half-hourly, long format
print(len(gas), len(turkey), vic_elec["unique_id"].unique(), vic_elec["ds"].min(), vic_elec["ds"].max())

## [Slide 4] 12.1 Example: Weekly US Gasoline
*Figure code (not shown on the slide).*

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(gas["ds"], gas["y"], "k", lw=0.7)
ax.set_xlabel("Year"); ax.set_ylabel("Million barrels per day")
ax.set_title("Weekly US finished motor gasoline products supplied")
fig.tight_layout()

## [Slide 5] 12.1 Example: Bank Call Volume

In [ ]:
bank_calls = (
    pd.read_csv(BASE + "bank_calls.csv", parse_dates=["ds"])
    .query("y > 0")
    .pipe(fill_gaps, freq="5min")  # NaN rows when closed
)
print(len(bank_calls), "rows after fill_gaps;",
      bank_calls["y"].notna().sum(), "observations")

## [Slide 6] 12.1 Bank Call Volume (Figure 12.1)
*Figure code (not shown on the slide).*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(bank_calls["ds"], bank_calls["y"], "k", lw=0.5)
ax.set_title("Five-minute call volume to bank"); ax.set_ylabel("Calls")
fig.tight_layout()

## [Slide 7] 12.1 Bank Calls: First Four Weeks (Figure 12.2)

In [ ]:
four_weeks = bank_calls.query("ds <= '2003-03-29'")

# figure code (not shown on the slide)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(four_weeks["ds"], four_weeks["y"], "k", lw=0.7)
ax.set_title("Five-minute call volume to bank over 4 weeks")
ax.set_ylabel("Calls")
fig.tight_layout()

## [Slide 8] 12.1 Example: Turkish Electricity Demand
*Figure code (not shown on the slide).*

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(turkey.index, turkey.values, "k", lw=0.5)
ax.set_xlabel("Year"); ax.set_ylabel("Electricity demand (GW)")
ax.set_title("Turkish daily electricity demand")
fig.tight_layout()

## [Slide 9] 12.1 MSTL with Multiple Seasonal Periods

In [ ]:
bank_calls = bank_calls.dropna()   # no NaN rows for fitting
model = MSTL(season_length=[169, 169 * 5])
sf = StatsForecast(models=[model], freq="5min")
sf = sf.fit(bank_calls.assign(y=lambda df: np.sqrt(df["y"])))
dcmp = sf.fitted_[0, 0].model_
# columns: data, trend, seasonal169, seasonal845, remainder
dcmp.head()

## [Slide 10] 12.1 MSTL Decomposition (Figure 12.3)
*Figure code (not shown on the slide).*

In [ ]:
fig, axes = plt.subplots(len(dcmp.columns), 1, figsize=(8, 6), sharex=True)
for ax, col in zip(axes, dcmp.columns):
    ax.plot(bank_calls["ds"].to_numpy(), dcmp[col].to_numpy(), "k", lw=0.5)
    ax.set_ylabel(col)
axes[0].set_title("sqrt(Calls) = trend + seasonal169 + seasonal845 + remainder")
fig.tight_layout()

## [Slide 11] 12.1 MSTL Forecasts: Future Timestamps

In [ ]:
start = bank_calls["ds"].max() + pd.offsets.Minute(5)
all_5min = pd.date_range(start, start + pd.offsets.Day(7),
                         freq="5min")
fc_ds = all_5min.to_series().loc[lambda x: (
    x.dt.weekday.between(0, 4)                      # Mon-Fri
    & x.dt.strftime("%H:%M").between("07:00", "21:00")
)]
print(len(fc_ds))   # 845 = 169 x 5

fc = sf.predict(h=len(fc_ds), level=[80, 95])
res = fc.select_dtypes(include=np.number)
fc = fc.assign(**res.transform(lambda x: x**2),  # square
               ds=fc_ds.to_numpy())              # real times
fc_mstl = fc.copy()
fc.head()

## [Slide 12] 12.1 MSTL Forecasts (Figure 12.4)
*Figure code (not shown on the slide).*

In [ ]:
from matplotlib.ticker import MaxNLocator

def plot_fc_5min(hist, fc, col, title, n_insample=169 * (14 + 5)):
    # last 19 days of data + forecasts, gaps (nights, weekends) left blank
    h = hist.tail(n_insample)
    hg = fill_gaps(h, freq="5min"); fg = fill_gaps(fc, freq="5min")
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.plot(hg["ds"], hg["y"], "k", lw=0.6, label="y")
    ax.plot(fg["ds"], fg[col], "C0", lw=0.8, label=col)
    for lv, a in [(95, 0.15), (80, 0.3)]:
        ax.fill_between(fg["ds"], fg[f"{col}-lo-{lv}"], fg[f"{col}-hi-{lv}"],
                        color="C0", alpha=a, label=f"{lv}% PI")
    ax.set_title(title); ax.legend(loc="upper left", ncol=4, fontsize=8)
    ax.xaxis.set_major_locator(MaxNLocator(4))
    fig.tight_layout()

plot_fc_5min(bank_calls, fc_mstl, "MSTL", "MSTL forecasts: call volume")

## [Slide 15] 12.1 Dynamic Harmonic Regression — Code

In [ ]:
features = [
    partial(fourier, season_length=169, k=10),
    partial(fourier, season_length=5 * 169, k=5),
]
bank_fourier, bank_futr = pipeline(
    bank_calls, features=features, freq="5min", h=len(fc_ds)
)
dropcols = ["sin5_845", "cos5_845"]  # = k=1 of 169
bank_fourier = bank_fourier.drop(columns=dropcols)
bank_futr = bank_futr.drop(columns=dropcols)

model = AutoARIMA(d=0, seasonal=False, nmodels=5)
sf = StatsForecast(models=[model], freq="5min")
sf = sf.fit(bank_fourier)
fc_dhr = sf.predict(
    h=len(fc_ds), X_df=bank_futr, level=[80, 95]
).assign(ds=fc_ds.to_numpy())

arima = sf.fitted_[0, 0].model_
print("ARMA order:", arima["arma"],
      "| number of coefficients:", len(arima["coef"]))   # ARMA(2,2), 33

## [Slide 16] 12.1 DHR Forecasts (Figure 12.5)
*Figure code (not shown on the slide).*

In [ ]:
plot_fc_5min(bank_calls, fc_dhr, "AutoARIMA",
             "Dynamic harmonic regression forecasts: call volume")

## [Slide 18] 12.1 Example: Electricity Demand (Figure 12.6)
*Figure code (not shown on the slide).*

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
for ax, uid, yl in zip(axes, ["Demand", "Temperature"],
                       ["Demand (MWh)", "Temperature (°C)"]):
    d = vic_elec.query("unique_id == @uid")
    ax.plot(d["ds"], d["y"], "k", lw=0.3); ax.set_ylabel(yl)
axes[0].set_title("Half-hourly electricity demand and temperature: Victoria")
fig.tight_layout()

## [Slide 19] 12.1 Electricity Demand: Data Preparation

In [ ]:
vic_elec = pd.read_csv(BASE + "vic_elec.csv",
                       parse_dates=["ds"])
vic_elec_df = (
    vic_elec.query("ds >= '2014'")
    .pivot(index=["ds", "Holiday"], columns="unique_id",
           values="y")
    .reset_index()               # wide: Demand, Temperature
    .assign(
        Demand=lambda df: df["Demand"] / 1e3,          # GW
        weekend=lambda df: df["ds"].dt.dayofweek.isin([5, 6]),
        Working_Day=lambda df: (~df["Holiday"]
                                & ~df["weekend"]),
        Cooling=lambda df: df["Temperature"].clip(lower=18),
        unique_id="VIC",
    )
    .drop(columns=["weekend", "Holiday"])
)
# columns: ds, Demand, Temperature, Working_Day, Cooling,
#          unique_id
vic_elec_df.columns.name = None
print(vic_elec_df.shape)
vic_elec_df.head()

## [Slide 20] 12.1 Demand vs. Temperature (Figure 12.7)
*Figure code (not shown on the slide).*

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for wd, c in [(True, "C0"), (False, "C1")]:
    d = vic_elec_df[vic_elec_df["Working_Day"] == wd]
    ax.scatter(d["Temperature"], d["Demand"], s=4, alpha=0.4, color=c,
               label=f"Working_Day = {wd}")
ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Electricity demand (GW)")
ax.legend()
fig.tight_layout()

## [Slide 21] 12.1 Electricity Demand: DHR Model — Code

In [ ]:
# NOTE: this cell takes about 20 minutes (AutoARIMA with 39 regressors)
features = [
    partial(fourier, season_length=2 * 24, k=10),      # day
    partial(fourier, season_length=2 * 24 * 7, k=5),   # week
    partial(fourier, season_length=2 * 24 * 365, k=3), # year
]
vic_fourier, vic_futr = pipeline(
    vic_elec_df, features=features, freq="30min", h=2 * 48
)
model = AutoARIMA(max_d=0, seasonal=False, nmodels=20)
sf = StatsForecast(models=[model], freq="30min")
sf = sf.fit(vic_fourier, target_col="Demand")
# predictors: Temperature, Working_Day, Cooling and
#   36 sin/cos columns (2 x (10 + 5 + 3))

arima = sf.fitted_[0, 0].model_
print("ARMA order:", arima["arma"], "| coefficients:", len(arima["coef"]))

## [Slide 22] 12.1 Electricity Demand: Forecasts (Figure 12.8)

In [ ]:
futr_df = (
    vic_elec_df.tail(2 * 48)
    .assign(ds=lambda df: df["ds"] + pd.Timedelta("2d"))
    .drop(columns=["Demand"])
    .merge(vic_futr, on=["unique_id", "ds"])  # + Fourier
)
fc = sf.predict(h=len(futr_df), X_df=futr_df, level=[80, 95])

# figure code (not shown on the slide)
hist = vic_elec_df.tail(10 * 48)
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(hist["ds"], hist["Demand"], "k", lw=0.8, label="Demand")
ax.plot(fc["ds"], fc["AutoARIMA"], "C0", lw=0.9, label="AutoARIMA")
for lv, a in [(95, 0.15), (80, 0.3)]:
    ax.fill_between(fc["ds"], fc[f"AutoARIMA-lo-{lv}"], fc[f"AutoARIMA-hi-{lv}"],
                    color="C0", alpha=a, label=f"{lv}% PI")
ax.set_title("Half hourly electricity demand: Victoria"); ax.set_ylabel("Demand (GW)")
ax.legend(loc="upper left", ncol=4, fontsize=8)
fig.tight_layout()

## [Slide 23] 12.1 Electricity Demand: Residuals (Figure 12.9)

In [ ]:
residuals = sf.fitted_[0, 0].model_["residuals"]

# figure code (not shown on the slide)
fig, axd = plt.subplot_mosaic([["r", "r"], ["a", "h"]], figsize=(8, 4.5))
axd["r"].plot(vic_elec_df["ds"].to_numpy(), residuals, "k", lw=0.4)
axd["r"].set_title("Innovation residuals")
plot_acf(residuals, ax=axd["a"], lags=48, zero=False, auto_ylims=True,
         bartlett_confint=False, title="ACF")
axd["h"].hist(residuals, bins=30, color="gray"); axd["h"].set_title("Histogram")
fig.tight_layout()

## [Slide 27] 12.1 TBATS in statsforecast: Weekly Gasoline

In [ ]:
gas = pd.read_csv(BASE + "us_gasoline.csv", parse_dates=["ds"])
train = gas.query("ds < '2015'")
sf = StatsForecast(models=[AutoTBATS(season_length=[52])],
                   freq="W-MON")
fc = sf.forecast(df=train, h=104, level=[80, 95])

# figure code (not shown on the slide)
fig, ax = plt.subplots(figsize=(8, 3.2))
h = gas.query("ds >= '2008'")
ax.plot(h["ds"], h["y"], "k", lw=0.7, label="y")
ax.plot(fc["ds"], fc["AutoTBATS"], "C0", lw=0.9, label="AutoTBATS")
for lv, a in [(95, 0.15), (80, 0.3)]:
    ax.fill_between(fc["ds"], fc[f"AutoTBATS-lo-{lv}"], fc[f"AutoTBATS-hi-{lv}"],
                    color="C0", alpha=a, label=f"{lv}% PI")
ax.set_title("AutoTBATS forecasts: weekly US gasoline (training data to 2014)")
ax.legend(loc="upper left", ncol=4, fontsize=8)
fig.tight_layout()

## [Slide 29] 12.2 Example: Cement Production — Code

In [ ]:
from prophet import Prophet
aus_production = pd.read_csv(BASE + "aus_production.csv",
                             parse_dates=["ds"])
cement = (aus_production.query("ds >= '1988'")
          [["ds", "Cement"]].rename(columns={"Cement": "y"})
          .assign(unique_id="Cement"))
train = cement.query("ds < '2008'")
test = cement.query("ds >= '2008'")

m = Prophet(yearly_seasonality=2)   # order 2 (default 10)
m.fit(train)                        # no weekly/daily terms
future = m.make_future_dataframe(periods=10, freq="QS")
fc_prophet = (
    m.predict(future)
    [["ds", "yhat", "yhat_lower", "yhat_upper"]]
    .rename(columns={"yhat": "Prophet",
                     "yhat_lower": "Prophet-lo-80",
                     "yhat_upper": "Prophet-hi-80"})
)
models = [AutoARIMA(season_length=4),
          AutoETS(season_length=4)]
sf = StatsForecast(models=models, freq="QS")
fc_stats = sf.forecast(df=train, h=10, level=[80])
fc = fc_stats.merge(fc_prophet, on="ds")
fc.head()

## [Slide 30] 12.2 Cement: Forecasts (Figure 12.10)

In [ ]:
# figure code (not shown on the slide)
def plot_fc_models(hist, fc, cols, title, ylabel, level=80):
    fig, ax = plt.subplots(figsize=(8, 3.3))
    ax.plot(hist["ds"], hist["y"], "k", lw=0.8, label="y")
    for i, col in enumerate(cols):
        ax.plot(fc["ds"], fc[col], f"C{i}", lw=1, label=col)
        ax.fill_between(fc["ds"], fc[f"{col}-lo-{level}"], fc[f"{col}-hi-{level}"],
                        color=f"C{i}", alpha=0.2)
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.legend(loc="upper left", ncol=len(cols) + 1, fontsize=8)
    fig.tight_layout()

plot_fc_models(cement, fc, ["AutoARIMA", "AutoETS", "Prophet"],
               "Cement production: Australia", "Cement (thousand tonnes)")

# accuracy on the test set (slide code)
evaluate(fc.merge(test, on=["unique_id", "ds"]), train_df=train,
         metrics=[rmse, mae, mape,
                  partial(mase, seasonality=4)])

## [Slide 31] 12.2 Example: Electricity Demand — Code

In [ ]:
vic_demand = (vic_elec.query("unique_id == 'Demand'")
              .drop(columns="Holiday").assign(unique_id="VIC")
              [["unique_id", "ds", "y"]])
train = vic_demand.iloc[:-48]; test = vic_demand.iloc[-48:]

m = Prophet(daily_seasonality=10, weekly_seasonality=5,
            yearly_seasonality=3)
m.fit(train)
future = m.make_future_dataframe(periods=48 * 2, freq="30min")
fc_prophet = (
    m.predict(future)
    [["ds", "yhat", "yhat_lower", "yhat_upper", "trend",
      "multiplicative_terms", "additive_terms"]]
    .rename(columns={"yhat": "Prophet",
                     "yhat_lower": "Prophet-lo-80",
                     "yhat_upper": "Prophet-hi-80"})
    .merge(train[["ds", "y"]], on="ds", how="left")
    .assign(resid=lambda df: df["y"] - df["Prophet"])
)
fc_prophet.tail()

## [Slide 32] 12.2 Prophet Components (Figure 12.11)
*Figure code (not shown on the slide).*

In [ ]:
comp = fc_prophet[["ds", "y", "trend", "multiplicative_terms",
                   "additive_terms", "resid"]].dropna()
fig, axes = plt.subplots(5, 1, figsize=(8, 7), sharex=True)
for ax, col in zip(axes, ["y", "trend", "multiplicative_terms",
                          "additive_terms", "resid"]):
    ax.plot(comp["ds"], comp[col], "k", lw=0.3); ax.set_ylabel(col, fontsize=8)
axes[0].set_title("Demand = trend * (1 + multiplicative_terms) + additive_terms + resid")
fig.tight_layout()

## [Slide 33] 12.2 Prophet Residuals (Figure 12.12)
*Figure code (not shown on the slide).*

In [ ]:
resid = fc_prophet[["ds", "resid"]].dropna()
fig, axd = plt.subplot_mosaic([["r", "r"], ["a", "h"]], figsize=(8, 4.5))
axd["r"].plot(resid["ds"], resid["resid"], "k", lw=0.3); axd["r"].set_title("Residuals")
plot_acf(resid["resid"], ax=axd["a"], lags=48, zero=False, auto_ylims=True,
         bartlett_confint=False, title="ACF")
axd["h"].hist(resid["resid"], bins=30, color="gray"); axd["h"].set_title("Histogram")
fig.tight_layout()

## [Slide 34] 12.2 Prophet vs. MSTL — Code

In [ ]:
sf = StatsForecast(models=[MSTL(season_length=[48, 48 * 7])],
                   freq="30min")
fc_stats = sf.forecast(df=train, h=48, level=[80])
cols = ["ds", "Prophet", "Prophet-lo-80", "Prophet-hi-80"]
fc = fc_stats.merge(fc_prophet[cols], on="ds")
evaluate(fc.merge(test, on=["unique_id", "ds"]), train_df=train,
         metrics=[rmse, mae, mape,
                  partial(mase, seasonality=48)])

## [Slide 35] 12.2 Prophet vs. MSTL: Forecasts (Figure 12.13)
*Figure code (not shown on the slide).*

In [ ]:
plot_fc_models(train.tail(48 * 7), fc, ["MSTL", "Prophet"],
               "Half-hourly electricity demand: Victoria", "Demand (MWh)")

## [Slide 37] 12.2 AutoARIMAProphet and NeuralProphet

In [ ]:
from statsforecast.adapters.prophet import AutoARIMAProphet
m = AutoARIMAProphet(daily_seasonality=10,
                     weekly_seasonality=5,
                     yearly_seasonality=3)
m.fit(train)
future = m.make_future_dataframe(periods=48 * 2, freq="30min")
fc_ap = m.predict(future)   # Prophet-style API (yhat, ...)

# accuracy on the same last-day test set (not shown on the slide)
fc2 = fc.merge(fc_ap[["ds", "yhat"]].rename(columns={"yhat": "AutoARIMAProphet"}),
               on="ds")
evaluate(fc2[["unique_id", "ds", "MSTL", "Prophet", "AutoARIMAProphet"]]
         .merge(test, on=["unique_id", "ds"]), train_df=train,
         metrics=[rmse, mae, mape, partial(mase, seasonality=48)])

## [Slide 43] 12.3 Example: US Consumption — Code

In [ ]:
from statsmodels.tsa.api import VAR
us_change = (
    pd.read_csv(BASE + "US_change.csv", parse_dates=["ds"])
    .rename(columns={"y": "Consumption"})
    .drop(columns=["unique_id"])
    .set_index("ds").asfreq("QS")
)[["Consumption", "Income"]]                # K = 2

fit_aic = VAR(us_change).fit(ic="aic")
fit_bic = VAR(us_change).fit(ic="bic")
print("Lags selected by AIC:", fit_aic.k_ar)   # 5
print("Lags selected by BIC:", fit_bic.k_ar)   # 1

## [Slide 44] 12.3 Residual ACF (Figure 12.14)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 4.5))
fits = [("VAR(5)", fit_aic), ("VAR(1)", fit_bic)]
for i, (name, fit) in enumerate(fits):
    for j, col in enumerate(["Consumption", "Income"]):
        plot_acf(fit.resid[col], ax=axes[i, j], lags=24,
                 zero=False, auto_ylims=True,
                 bartlett_confint=False,
                 title=f"{name}: {col}")
fig.tight_layout()

## [Slide 45] 12.3 Forecasting — Code

In [ ]:
lag_order = fit_aic.k_ar                    # 5
y = us_change.tail(lag_order).to_numpy()    # last p obs.
fc80 = fit_aic.forecast_interval(y, steps=8, alpha=0.2)
fc95 = fit_aic.forecast_interval(y, steps=8, alpha=0.05)
# each: (mean, lower, upper), arrays of shape (steps, K)

fc_var5 = pd.concat([
    pd.DataFrame({
        "unique_id": column,
        "ds": pd.date_range("2019-07-01", periods=8,
                            freq="QS"),
        "VAR_5": fc80[0][:, i],
        "VAR_5-lo-80": fc80[1][:, i],
        "VAR_5-hi-80": fc80[2][:, i],
        "VAR_5-lo-95": fc95[1][:, i],
        "VAR_5-hi-95": fc95[2][:, i],
    })
    for i, column in enumerate(["Consumption", "Income"])
])
fc_var5.round(3)

## [Slide 46] 12.3 Forecasts from the VAR(5) (Figure 12.15)
*Figure code (not shown on the slide).*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
for ax, col in zip(axes, ["Consumption", "Income"]):
    h = us_change[col].tail(4 * 20); f = fc_var5.query("unique_id == @col")
    ax.plot(h.index, h.values, "k", lw=0.8, label=col)
    ax.plot(f["ds"], f["VAR_5"], "C0", lw=1, label="VAR_5")
    for lv, a in [(95, 0.15), (80, 0.3)]:
        ax.fill_between(f["ds"], f[f"VAR_5-lo-{lv}"], f[f"VAR_5-hi-{lv}"],
                        color="C0", alpha=a)
    ax.set_title(col)
fig.supxlabel("Quarter")
fig.tight_layout()

## [Slide 47] 12.3 Granger Causality

In [ ]:
print(fit_aic.test_causality("Consumption", ["Income"], kind="f").summary())
# F = 1.65, p = 0.147: Income does not G-cause Consumption
print(fit_aic.test_causality("Income", ["Consumption"], kind="f").summary())
# F = 9.89, p < 0.001: Consumption Granger-causes Income

## [Slide 50] 12.4 STL Decomposition of Cement (Figure 12.16)

In [ ]:
sf = StatsForecast(models=[MSTL(season_length=4)], freq="QS")
sf = sf.fit(cement)
dcmp = sf.fitted_[0, 0].model_
dcmp = dcmp.assign(ds=cement["ds"].to_numpy())
# columns: data, trend, seasonal, remainder, ds

# figure code (not shown on the slide)
fig, axes = plt.subplots(4, 1, figsize=(8, 5.5), sharex=True)
for ax, col in zip(axes, ["data", "trend", "seasonal", "remainder"]):
    ax.plot(dcmp["ds"], dcmp[col], "k", lw=0.8); ax.set_ylabel(col)
axes[0].set_title("Cement = trend + seasonal + remainder")
fig.tight_layout()

## [Slide 51] 12.4 Blocked Bootstrap — Code

In [ ]:
def blocked_bootstrap(dcmp, block_size, num_sims):
    num_blocks = int(np.ceil(len(dcmp) / block_size))  # 12
    sim = {}
    for i in range(num_sims):
        blocks = []
        for _ in range(num_blocks):
            start = np.random.randint(
                0, len(dcmp) - block_size + 1)
            blocks.append(
                dcmp["remainder"].iloc[start:start+block_size]
            )
        sim[str(i)] = (
            dcmp["trend"] + dcmp["seasonal"]
            + pd.concat(blocks, ignore_index=True)[:len(dcmp)]
        )
    return pd.DataFrame(sim)   # one column per series

sim_df = blocked_bootstrap(dcmp, block_size=8, num_sims=10)
sim_df = sim_df.assign(unique_id="Cement",
                       ds=cement["ds"].to_numpy())
sim_df.head()

## [Slide 52] 12.4 Ten Bootstrapped Series (Figure 12.17)
*Figure code (not shown on the slide).*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.3))
for i in range(10):
    ax.plot(sim_df["ds"], sim_df[str(i)], lw=0.7, alpha=0.8)
ax.plot(cement["ds"], cement["y"], "k", lw=1.2, label="original")
ax.set_title("Cement production: 10 bootstrapped series (block size 8)")
ax.legend()
fig.tight_layout()

## [Slide 53] 12.4 Bagged Forecasts

In [ ]:
sim_df = blocked_bootstrap(dcmp, block_size=8, num_sims=100)
sim_df = sim_df.assign(unique_id="Cement",
                       ds=cement["ds"].to_numpy())
sf = StatsForecast(models=[AutoETS(season_length=4)],
                   freq="QS")
forecasts = []
for col in sim_df.drop(columns=["ds", "unique_id"]).columns:
    train = (sim_df[["unique_id", "ds", col]]   # "0".."99"
             .rename(columns={col: "y"}))
    fc = (sf.forecast(df=train, h=10)
          .rename(columns={"AutoETS": f"AutoETS_{col}"}))
    forecasts.append(fc)
forecasts_df = reduce(
    lambda left, right: left.merge(right,
                                   on=["unique_id", "ds"]),
    forecasts
)   # 10 rows x (2 + 100) columns
print(forecasts_df.shape)

## [Slide 54] 12.4 100 Bootstrapped Forecasts (Figure 12.18)
*Figure code (not shown on the slide).*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.3))
h = cement.query("ds >= '2000'")
for col in forecasts_df.columns[2:]:
    ax.plot(forecasts_df["ds"], forecasts_df[col], lw=0.5, alpha=0.5)
ax.plot(h["ds"], h["y"], "k", lw=1.2)
ax.set_title("Cement production: bootstrapped forecasts (100 AutoETS models)")
fig.tight_layout()

## [Slide 55] 12.4 Bagged vs. Direct Forecasts (Figure 12.19)

In [ ]:
fc_bagged = (forecasts_df.set_index(["unique_id", "ds"])
             .mean(axis="columns")       # average of 100
             .rename("AvgAutoETS").reset_index())
fc_ets = sf.forecast(df=cement, h=10)   # original data

# figure code (not shown on the slide)
fc = fc_ets.merge(fc_bagged, on=["unique_id", "ds"])
fig, ax = plt.subplots(figsize=(8, 3.3))
ax.plot(h["ds"], h["y"], "k", lw=1, label="y")
ax.plot(fc["ds"], fc["AutoETS"], "C0", lw=1.2, label="AutoETS")
ax.plot(fc["ds"], fc["AvgAutoETS"], "C1", lw=1.2, label="AvgAutoETS (bagged)")
ax.set_title("Cement production: bagged vs. direct AutoETS forecasts")
ax.legend()
fig.tight_layout()
fc.round(1)